# NB0 - Analytical A-S Baselines

Loads the fair train-only B0 results and the optional plain analytical A-S diagnostic. B0 is produced by `scripts/snellius/run_b0_fair.py` inside the Snellius pipeline.

## Snellius Pipeline Mode

This notebook is now a reporting/orchestration notebook. Heavy training and per-day evaluation are executed by Slurm scripts under `scripts/snellius/`; this notebook reads the resulting artifacts.

Expected artifacts:
- `results/b0_test_results.csv`
- `results/b0_gamma_fixed.txt`
- `results/b0_train_calibration_params.csv`
- `results/plain_as_test_results.csv`

Run from MobaXterm/Snellius with:

```bash
cd /home/<user>/thesis
export PROJECT_DIR=/home/<user>/thesis
export DATA_DIR=/scratch-shared/<user>/datasets
export CONDA_ENV=mysimenv
bash scripts/snellius/submit_final_pipeline.sh
```

In [ ]:
import json
import pathlib
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"
META_DIR = RESULTS_DIR / "job_metadata"

pd.set_option("display.float_format", "{:.6f}".format)


def require_file(path, label=None, strict=False):
    path = pathlib.Path(path)
    if path.exists():
        print(f"OK      {label or path.name}: {path}")
        return True
    message = f"MISSING {label or path.name}: {path}"
    if strict:
        raise FileNotFoundError(message)
    print(message)
    return False


def read_csv(path, **kwargs):
    path = pathlib.Path(path)
    require_file(path, strict=True)
    return pd.read_csv(path, **kwargs)


def metric_summary(df):
    metrics = [c for c in ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL", "Mean |q|", "Near Cap Fraction"] if c in df.columns]
    return pd.DataFrame({"Mean": df[metrics].mean(), "Median": df[metrics].median(), "Std": df[metrics].std(ddof=1)})


def formula_metadata_files():
    if not META_DIR.exists():
        return []
    return sorted(META_DIR.glob("*.json"))

print(f"Project root : {PROJECT_ROOT}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Models dir   : {MODELS_DIR}")
print("Official Snellius execution path: bash scripts/snellius/submit_final_pipeline.sh")

## Artifact Preflight

In [ ]:
require_file(RESULTS_DIR / 'b0_test_results.csv')
require_file(RESULTS_DIR / 'b0_gamma_fixed.txt')
require_file(RESULTS_DIR / 'b0_train_calibration_params.csv')
require_file(RESULTS_DIR / 'plain_as_test_results.csv')

## Load B0 Results

In [ ]:
df_b0 = read_csv(RESULTS_DIR / "b0_test_results.csv")
display(df_b0.head())
display(metric_summary(df_b0))

if (RESULTS_DIR / "plain_as_test_results.csv").exists():
    df_plain = read_csv(RESULTS_DIR / "plain_as_test_results.csv")
    print("Plain A-S summary")
    display(metric_summary(df_plain))

## B0 Metric Plot

In [ ]:
metrics = [m for m in ["Sharpe", "Sortino", "Max DD", "Final PnL"] if m in df_b0]
axes = df_b0.set_index("Day")[metrics].plot(subplots=True, figsize=(12, 8), layout=(2, 2), legend=False, title=metrics)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "nb0_b0_report_plot.png", dpi=150, bbox_inches="tight")
plt.show()